# Immune age in HNSCC and NSCLC immunotherapy cohorts

Paper: Fig. 6f–h. Inputs: final trained PeriClock artifact and two original annotated cohort AnnData files, with approved clinical metadata. Environment: Python/Scanpy, NumPy, pandas, SciPy, scikit-learn, joblib, tqdm and Matplotlib. Outputs: original clinical prediction CSV and response plots, below `OUTPUT_DIR`. No earlier tutorial memory is needed. Annotation is an **external prerequisite**, not silently rerun: HNSCC uses CellTypist; NSCLC uses Symphony. The package usage example explains configurable annotation resources. These restricted cohort inputs and results are not bundled; execution and paper-level numeric reproduction remain unverified.

Original source indices below are **zero-based**. Original cell hashes are retained in cell metadata. Outputs were cleared; inclusion does not imply successful execution.

## Required resources and configuration

Final PeriClock model, approved prepared HNSCC/NSCLC annotated AnnData including required clinical metadata, and licensed Arial fonts; [immunotherapy data](../RESOURCE_DOWNLOAD.md#7-immunotherapy-datasets). No raw-cohort download is an automatic substitute.

Download locations, sizes, SHA256, versions, permissions and publication status are maintained once in [RESOURCE_DOWNLOAD.md](../RESOURCE_DOWNLOAD.md). Individual GEO/GSM sources and sample choices are in the paper’s Supplementary Table 1. Clock models are uploaded to [Hugging Face Models](https://huggingface.co/LeyiZhang1/PeriClock); [Dataset uploads](https://huggingface.co/datasets/LeyiZhang1/PeriClock-data) are in progress. Both repositories are currently private. Consult the resource guide for per-file availability; donor-level public redistribution remains subject to review.

Set `PERICLOCK_RESOURCE_DIR` for external data and `PERICLOCK_MODEL_DIR` for models, or override individual file paths in the next cell. Keep outputs separate from inputs. Start a fresh kernel and run in order.


In [ ]:
from pathlib import Path
import os
# Start Jupyter from within this GitHub checkout.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "periclock").is_dir() and (p / "pyproject.toml").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook kernel from within the PeriClock code checkout")
RESOURCE_DIR = Path(os.environ.get("PERICLOCK_RESOURCE_DIR", "~/periclock-resources")).expanduser().resolve()
MODEL_DIR = Path(os.environ.get("PERICLOCK_MODEL_DIR", str(RESOURCE_DIR / "models"))).expanduser().resolve()
# Each individual file-path environment variable still overrides these defaults.
MODEL_PATH = Path(os.environ.get("PERICLOCK_PERTURBATION_MODEL", str(MODEL_DIR / "PeriClock.pkl"))).expanduser().resolve()
HNSCC_H5AD = Path(os.environ.get("PERICLOCK_HNSCC_H5AD", str(RESOURCE_DIR / "datasets/HNSCC_Blood_celltypist_annotated.h5ad"))).expanduser().resolve()
NSCLC_H5AD = Path(os.environ.get("PERICLOCK_NSCLC_H5AD", str(RESOURCE_DIR / "datasets/GSE306542_Symphony_annotated.h5ad"))).expanduser().resolve()
FONT_DIR = Path(os.environ.get("PERICLOCK_FONT_DIR", "/path/to/Arial")).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get("PERICLOCK_TUTORIAL_OUTPUT", str(PROJECT_ROOT / "outputs/09_immunotherapy"))).expanduser().resolve()
# Input files remain read-only; all relative output paths below use this directory.
for input_path in [MODEL_PATH, HNSCC_H5AD, NSCLC_H5AD]:
    if not input_path.is_file():
        raise FileNotFoundError(f"Set the input path in this cell: {input_path}")
if not FONT_DIR.is_dir():
    raise FileNotFoundError("Set PERICLOCK_FONT_DIR to a directory containing licensed Arial fonts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(OUTPUT_DIR)

## HNSCC baseline immune age (Fig. 6f)

Source: `HNSCC/exp/hnscc.ipynb`, cells 69–72, 74 and 76. Start from the original QC-processed, log1p(CP10K), CellTypist-annotated blood dataset, with `.obs` fields `sample`, `patient`, `timepoint`, `age`, `MPR`, `pathologic_response` and `celltypist_pred`. The original P0 selection, unknown-label exclusion, predictor, metadata join and permutation test are retained. Earlier code containing a patient-level clinical dictionary is deliberately not distributed. You must obtain the approved annotated input separately; no clinical values are reconstructed here.

In [ ]:
import scanpy as sc

In [ ]:
import joblib
pipeline = joblib.load(str(MODEL_PATH))

def predict_age_with_model(adata, pipeline, individual_col, age_col, celltype_col, strain_col=None):
    """
    基因模型预测
    """
    from tqdm import tqdm
    import numpy as np
    import pandas as pd

    vt = pipeline['vt']
    skb = pipeline['skb']
    scaler_prop = pipeline['scaler_prop']
    scaler_expr = pipeline['scaler_expr']
    model = pipeline['model']
    prop_cols = pipeline['prop_cols']
    expr_cols = pipeline['expr_cols']
    selected_features = pipeline['selected_features']
    
    # 1. 构建细胞类型比例矩阵
    cell_counts = adata.obs.groupby([individual_col, age_col, celltype_col]).size().unstack(fill_value=0)
    cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0).dropna(axis=0, how='any')

    df_valid = cell_props.reset_index()
    df_valid[individual_col] = df_valid[individual_col].astype(str)
    df_valid[individual_col] = df_valid.groupby(individual_col).cumcount().astype(str).radd(df_valid[individual_col] + '_')
    df_valid.set_index(individual_col, inplace=True)


    # 去除细胞类型数量过少的个体
    zero_counts = (df_valid.iloc[:, 2:] == 0).sum(axis=1)
    df_valid = df_valid[zero_counts < 31]
    remaining_ids = df_valid.index.tolist()

    # Step 2: 构建特征矩阵
    genes = adata.var_names.tolist()
    gene_matrix = adata.X.A if hasattr(adata.X, "A") else adata.X
    
    data_rows = []
    index_names = []

    for ind in tqdm(remaining_ids):
        donor_mask = adata.obs[individual_col].astype(str) == ind.rsplit('_', 1)[0]
        adata_donor = adata[donor_mask]
        total_cells = adata_donor.n_obs

        row = {}
        try:
            row['Age'] = adata_donor.obs[age_col].iloc[0]
        except IndexError:
            continue

        for ct in adata_donor.obs[celltype_col].unique():
            ct_mask = donor_mask & (adata.obs[celltype_col] == ct)
            ct_cells = adata[ct_mask]
            ratio = ct_cells.n_obs / total_cells
            row[f"{ct}_ratio"] = ratio

            if ct_cells.n_obs == 0:
                gene_vals = np.zeros(len(genes))
            else:
                sub_expr = gene_matrix[ct_mask]
                gene_vals = sub_expr.mean(axis=0).A1 if hasattr(sub_expr, "A1") else np.asarray(sub_expr.mean(axis=0)).flatten()

            gene_names = [f"{ct}_{g}" for g in genes]
            row.update(dict(zip(gene_names, gene_vals)))

        data_rows.append(row)
        index_names.append(ind)

    df_features = pd.DataFrame(data_rows, index=index_names)
    df_features.index.name = individual_col
    df_features.fillna(0, inplace=True)     # 由于缺少某些细胞类型而出现的NaN用0补齐    
    
    # Step 3: 特征选择和模型预测
    y_valid = df_features['Age'].values
    X_valid = df_features.drop(columns='Age')

    # 填补缺失的列
    missing_cols = {col: 0.0 for col in prop_cols + expr_cols if col not in X_valid.columns}
    if missing_cols:
        X_valid = pd.concat([X_valid, pd.DataFrame(missing_cols, index=X_valid.index)], axis=1)

    X_valid = X_valid[prop_cols + expr_cols]

    X_prop = scaler_prop.transform(X_valid[prop_cols])
    X_expr = vt.transform(X_valid[expr_cols])
    X_expr = skb.transform(X_expr)
    X_expr = scaler_expr.transform(X_expr)
    X_final = np.hstack([X_prop, X_expr])

    # 4. 构建完整的特征名列表（顺序要和X_final一致）
    prop_feature_names = list(prop_cols)  # 确保是 list 而不是 numpy array
    expr_feature_names = list(skb.get_feature_names_out(vt.get_feature_names_out()))

    final_feature_names = prop_feature_names + expr_feature_names

    # 5. 根据 selected_features 找出对应的列索引
    selected_indices = [final_feature_names.index(f) for f in selected_features]
    print(len(selected_indices))

    # 6. 从 X_final 中挑选这些模型输入特征列
    X_selected = X_final[:, selected_indices]
    X_selected.shape
    
    y_pred = model.predict(X_selected)
    df_results = pd.DataFrame({
        'Actual_Age': y_valid,
        'Predicted_Age': y_pred,
        'Age_Diff': y_pred - y_valid
    }, index=df_features.index)


    return df_results

In [ ]:
adata = sc.read_h5ad(str(HNSCC_H5AD))
# ============================================================
# 1. 提取基线 P0 PBMC
# ============================================================
adata_p0 = adata[adata.obs["timepoint"] == "P0"].copy()

# ============================================================
# 2. 去掉 unknown
# ============================================================
adata_p0 = adata_p0[
    adata_p0.obs["celltypist_pred"].notna() &
    (adata_p0.obs["celltypist_pred"].astype(str) != "unknown")
].copy()

print("\nAfter removing unknown:")
print(adata_p0)
print(adata_p0.obs["sample"].value_counts())

In [ ]:
predict_result_df = predict_age_with_model(adata_p0, pipeline, 'sample', 'age', 'celltypist_pred', strain_col=None)

In [ ]:
# ============================================================
# 4. 整理预测结果 + 临床信息
# ============================================================
df_p0_pred = predict_result_df.copy()
df_p0_pred["sample"] = df_p0_pred.index.str.rsplit("_", n=1).str[0]

meta_p0 = adata_p0.obs[["sample", "patient", "age", "MPR", "pathologic_response"]].drop_duplicates("sample").set_index("sample")

df_p0_pred = df_p0_pred.set_index("sample")
df_p0_pred = df_p0_pred.join(meta_p0[["patient", "MPR", "pathologic_response"]])

df_p0_pred = df_p0_pred[[
    "patient",
    "Actual_Age",
    "Predicted_Age",
    "Age_Diff",
    "MPR",
    "pathologic_response"
]]

print(df_p0_pred)

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib as mpl, matplotlib.font_manager as fm
from scipy.stats import permutation_test

# =========================
# 1. 输出路径 + Arial
# =========================
outdir = str(OUTPUT_DIR / "HNSCC")
os.makedirs(outdir, exist_ok=True)

font_dir = str(FONT_DIR)
if os.path.exists(font_dir):
    for f in os.listdir(font_dir):
        if f.lower().endswith((".ttf", ".otf")): fm.fontManager.addfont(os.path.join(font_dir, f))

mpl.rcParams.update({
    "font.family": "Arial", "font.sans-serif": ["Arial"], "font.size": 13, "font.weight": "bold",
    "axes.labelsize": 14, "axes.labelweight": "bold", "axes.titlesize": 15, "axes.titleweight": "bold",
    "xtick.labelsize": 13, "ytick.labelsize": 13, "axes.linewidth": 1.2,
    "pdf.fonttype": 42, "ps.fonttype": 42
})

# =========================
# 2. 整理数据
# =========================
plot_df = df_p0_pred.copy().reset_index().rename(columns={"index": "sample"})
plot_df["Group"] = plot_df["MPR"].map({"Yes": "MPR", "No": "nMPR"})
plot_df["Label"] = plot_df["sample"]

order = ["MPR", "nMPR"]
xmap = {"MPR": 0, "nMPR": 1}
plot_df["x"] = plot_df["Group"].map(xmap).astype(float)

# 为了点不重叠，手动加一点轻微抖动
rng = np.random.RandomState(0)
plot_df["x_jitter"] = plot_df["x"] + rng.uniform(-0.08, 0.08, size=len(plot_df))

# =========================
# 新增：MPR vs nMPR 显著性检验
# =========================
mpr = plot_df.loc[plot_df["Group"] == "MPR", "Age_Diff"].dropna().to_numpy()
nmpr = plot_df.loc[plot_df["Group"] == "nMPR", "Age_Diff"].dropna().to_numpy()

if min(len(mpr), len(nmpr)) < 2:
    raise ValueError("至少一组不足2名患者，无法执行当前组间检验。")

res = permutation_test(
    (mpr, nmpr),
    statistic=lambda x, y: np.mean(x) - np.mean(y),
    permutation_type="independent",
    n_resamples=np.inf,
    alternative="two-sided"
)
p_value = res.pvalue
print(f"MPR (n={len(mpr)}) vs nMPR (n={len(nmpr)}): P = {p_value:.4f}")

# =========================
# 3. 画图
# =========================
fig, ax = plt.subplots(figsize=(4.8, 5.2))

group_colors = {"MPR": "#C77C8A", "nMPR": "#5B7F95"}

for grp in order:
    sub = plot_df[plot_df["Group"] == grp].copy()
    ax.scatter(sub["x_jitter"], sub["Age_Diff"], s=95, color=group_colors[grp], edgecolor="black", linewidth=0.8, zorder=3)
    med = sub["Age_Diff"].median()
    ax.hlines(med, xmap[grp]-0.18, xmap[grp]+0.18, colors="black", linewidth=2.0, zorder=2)

    for _, r in sub.iterrows():
        ax.text(r["x_jitter"] + 0.015, r["Age_Diff"] + 0.35, r["Label"], fontsize=11, fontweight="bold", ha="left", va="bottom")

ax.axhline(0, color="#666666", linestyle="--", linewidth=1.2, zorder=1)
ax.set_xticks([0, 1])
ax.set_xticklabels(order, fontweight="bold")
ax.set_ylabel("Age difference (Predicted - Actual)")
ax.set_xlabel("")
ax.set_title("Baseline PBMC immune age")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xlim(-0.35, 1.35)

# 让上下留一点边距
ymin, ymax = plot_df["Age_Diff"].min(), plot_df["Age_Diff"].max()
pad = max((ymax - ymin) * 0.18, 2.5)
ax.set_ylim(ymin - pad*0.35, ymax + pad)

# =========================
# 新增：显著性标注
# =========================
y_bracket = ymax + pad * 0.35
h = pad * 0.08

ax.plot(
    [0, 0, 1, 1],
    [y_bracket, y_bracket+h, y_bracket+h, y_bracket],
    color="black", linewidth=1.5
)
ax.text(
    0.5, y_bracket + h + pad*0.07,
    f"P = {p_value:.3f}",
    ha="center", va="bottom",
    fontsize=14, fontweight="bold"
)

plt.tight_layout()

pdf_file = os.path.join(outdir, "HNSCC_P0_MPR_vs_nMPR_AgeDiff_scatter_celltypist.pdf")
png_file = os.path.join(outdir, "HNSCC_P0_MPR_vs_nMPR_AgeDiff_scatter_celltypist.png")
plt.savefig(pdf_file, bbox_inches="tight")
plt.savefig(png_file, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:")
print(pdf_file)
print(png_file)

## NSCLC baseline age and response (Fig. 6g–h)

Source: `NSCLC/label_transfer_Symphony.ipynb`, cells 12–15, 18 and 22–23. Supply the original QC-processed, log1p(CP10K), Symphony-annotated dataset. Required `.obs` fields include `sample_id`, `age`, `symphony_pred` and `early_response_9weeks` (`Yes`/`No`); the source retains available patient/clinical metadata in its join. Original two-sided Mann–Whitney tests and group definitions remain unchanged. The predictor is redefined here exactly as in this source, before the NSCLC call. Outputs may contain clinical information: keep generated CSVs and executed notebooks private. Full reference mapping and clinical prediction have not been run during preparation.

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd

In [ ]:
adata = sc.read_h5ad(str(NSCLC_H5AD))
adata

In [ ]:
import joblib
pipeline = joblib.load(str(MODEL_PATH))

def predict_age_with_model(adata, pipeline, individual_col, age_col, celltype_col, strain_col=None):
    """
    基因模型预测
    """
    from tqdm import tqdm
    import numpy as np
    import pandas as pd

    vt = pipeline['vt']
    skb = pipeline['skb']
    scaler_prop = pipeline['scaler_prop']
    scaler_expr = pipeline['scaler_expr']
    model = pipeline['model']
    prop_cols = pipeline['prop_cols']
    expr_cols = pipeline['expr_cols']
    selected_features = pipeline['selected_features']
    
    # 1. 构建细胞类型比例矩阵
    cell_counts = adata.obs.groupby([individual_col, age_col, celltype_col]).size().unstack(fill_value=0)
    cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0).dropna(axis=0, how='any')

    df_valid = cell_props.reset_index()
    df_valid[individual_col] = df_valid[individual_col].astype(str)
    df_valid[individual_col] = df_valid.groupby(individual_col).cumcount().astype(str).radd(df_valid[individual_col] + '_')
    df_valid.set_index(individual_col, inplace=True)


    # 去除细胞类型数量过少的个体
    zero_counts = (df_valid.iloc[:, 2:] == 0).sum(axis=1)
    df_valid = df_valid[zero_counts < 31]
    remaining_ids = df_valid.index.tolist()

    # Step 2: 构建特征矩阵
    genes = adata.var_names.tolist()
    gene_matrix = adata.X.A if hasattr(adata.X, "A") else adata.X
    
    data_rows = []
    index_names = []

    for ind in tqdm(remaining_ids):
        donor_mask = adata.obs[individual_col].astype(str) == ind.rsplit('_', 1)[0]
        adata_donor = adata[donor_mask]
        total_cells = adata_donor.n_obs

        row = {}
        try:
            row['Age'] = adata_donor.obs[age_col].iloc[0]
        except IndexError:
            continue

        for ct in adata_donor.obs[celltype_col].unique():
            ct_mask = donor_mask & (adata.obs[celltype_col] == ct)
            ct_cells = adata[ct_mask]
            ratio = ct_cells.n_obs / total_cells
            row[f"{ct}_ratio"] = ratio

            if ct_cells.n_obs == 0:
                gene_vals = np.zeros(len(genes))
            else:
                sub_expr = gene_matrix[ct_mask]
                gene_vals = sub_expr.mean(axis=0).A1 if hasattr(sub_expr, "A1") else np.asarray(sub_expr.mean(axis=0)).flatten()

            gene_names = [f"{ct}_{g}" for g in genes]
            row.update(dict(zip(gene_names, gene_vals)))

        data_rows.append(row)
        index_names.append(ind)

    df_features = pd.DataFrame(data_rows, index=index_names)
    df_features.index.name = individual_col
    df_features.fillna(0, inplace=True)     # 由于缺少某些细胞类型而出现的NaN用0补齐    
    
    # Step 3: 特征选择和模型预测
    y_valid = df_features['Age'].values
    X_valid = df_features.drop(columns='Age')

    # 填补缺失的列
    missing_cols = {col: 0.0 for col in prop_cols + expr_cols if col not in X_valid.columns}
    if missing_cols:
        X_valid = pd.concat([X_valid, pd.DataFrame(missing_cols, index=X_valid.index)], axis=1)

    X_valid = X_valid[prop_cols + expr_cols]

    X_prop = scaler_prop.transform(X_valid[prop_cols])
    X_expr = vt.transform(X_valid[expr_cols])
    X_expr = skb.transform(X_expr)
    X_expr = scaler_expr.transform(X_expr)
    X_final = np.hstack([X_prop, X_expr])

    # 4. 构建完整的特征名列表（顺序要和X_final一致）
    prop_feature_names = list(prop_cols)  # 确保是 list 而不是 numpy array
    expr_feature_names = list(skb.get_feature_names_out(vt.get_feature_names_out()))

    final_feature_names = prop_feature_names + expr_feature_names

    # 5. 根据 selected_features 找出对应的列索引
    selected_indices = [final_feature_names.index(f) for f in selected_features]
    print(len(selected_indices))

    # 6. 从 X_final 中挑选这些模型输入特征列
    X_selected = X_final[:, selected_indices]
    X_selected.shape
    
    y_pred = model.predict(X_selected)
    df_results = pd.DataFrame({
        'Actual_Age': y_valid,
        'Predicted_Age': y_pred,
        'Age_Diff': y_pred - y_valid
    }, index=df_features.index)


    return df_results

In [ ]:
predict_result_df = predict_age_with_model(adata, pipeline, 'sample_id', 'age', 'symphony_pred', strain_col=None)

In [ ]:
import numpy as np
import pandas as pd

predict_result_df.index = predict_result_df.index.astype(str).str.replace(r"_0$", "", regex=True)
predict_result_df.index.name = "sample_id"

print(predict_result_df)

# ============================================================
# 1. 提取需要的临床信息
# ============================================================
clinical_cols = [
    "patient_ids",
    "sample_id",
    "gsm_id",
    "age",
    "sex",
    "percent_PD_L1",
    "histology",
    "mutations",
    "early_response_9weeks",
    "response_type_9weeks",
    "durable_response_6months",
    "irAE"
]

clinical_cols = [c for c in clinical_cols if c in adata.obs.columns]
clinical_obs = adata.obs[clinical_cols].copy()

# 避免 categorical 类型引起的后续操作问题
for col in clinical_obs.select_dtypes(include="category").columns:
    clinical_obs[col] = clinical_obs[col].astype("object")

# ============================================================
# 2. 检查同一样本临床信息是否一致
# ============================================================
check_cols = [c for c in clinical_cols if c != "sample_id"]

conflicts = clinical_obs.groupby("sample_id", dropna=False)[check_cols].nunique(dropna=True)
conflicts = conflicts[conflicts.gt(1).any(axis=1)]

if not conflicts.empty:
    print("WARNING: Conflicting metadata within samples:")
    print(conflicts.loc[:, conflicts.gt(1).any(axis=0)].to_string())
    raise ValueError("Clinical metadata conflicts detected. Please check before merging.")

# ============================================================
# 3. 构建样本级临床信息
# ============================================================
def first_valid(x):
    x = x.dropna()
    return x.iloc[0] if len(x) else np.nan

sample_meta = clinical_obs.groupby("sample_id", dropna=False)[check_cols].agg(first_valid)
sample_meta["n_cells"] = clinical_obs.groupby("sample_id", dropna=False).size()

sample_meta = sample_meta.rename(columns={"age": "Meta_Age"})
sample_meta.index = sample_meta.index.astype(str)
sample_meta.index.name = "sample_id"

print("Number of samples:", len(sample_meta))

# ============================================================
# 4. 合并 PeriClock 预测结果
# ============================================================
prediction_df = predict_result_df.copy()

if "sample_id" in prediction_df.columns:
    prediction_df = prediction_df.set_index("sample_id")

prediction_df.index = prediction_df.index.astype(str)
prediction_df.index.name = "sample_id"

if not prediction_df.index.is_unique:
    raise ValueError("predict_result_df contains duplicated sample_id.")

if not sample_meta.index.is_unique:
    raise ValueError("sample_meta contains duplicated sample_id.")

missing_samples = prediction_df.index.difference(sample_meta.index)

if len(missing_samples):
    raise ValueError(f"Samples not found in adata.obs: {missing_samples.tolist()}")

prediction_clinical_df = prediction_df.join(
    sample_meta,
    how="left",
    validate="one_to_one"
)

# ============================================================
# 5. 检查年龄与 Age_Diff
# ============================================================
actual = pd.to_numeric(prediction_clinical_df["Actual_Age"], errors="coerce")
meta_age = pd.to_numeric(prediction_clinical_df["Meta_Age"], errors="coerce")
pred_age = pd.to_numeric(prediction_clinical_df["Predicted_Age"], errors="coerce")
age_diff = pd.to_numeric(prediction_clinical_df["Age_Diff"], errors="coerce")

age_mismatch = actual.notna() & meta_age.notna() & ~np.isclose(actual, meta_age, atol=1e-6)
diff_mismatch = ~np.isclose(age_diff, pred_age - actual, atol=1e-6)

print("\nAge metadata mismatches:", age_mismatch.sum())
print("Age_Diff calculation mismatches:", diff_mismatch.sum())

# ============================================================
# 6. 调整列顺序
# ============================================================
ordered_cols = [
    "patient_ids", "Actual_Age", "Predicted_Age", "Age_Diff",
    "sex", "histology", "percent_PD_L1", "mutations",
    "early_response_9weeks", "response_type_9weeks",
    "durable_response_6months", "irAE",
    "n_cells", "gsm_id", "Meta_Age"
]

ordered_cols = [c for c in ordered_cols if c in prediction_clinical_df.columns]
prediction_clinical_df = prediction_clinical_df[ordered_cols]

# ============================================================
# 7. 展示并保存
# ============================================================
print("\n===== PeriClock prediction + clinical metadata =====")
print(prediction_clinical_df.to_string())

prediction_clinical_df.to_csv("GSE306542_PeriClock_clinical_results.csv")

print("\nSaved: GSE306542_PeriClock_clinical_results.csv")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
from scipy.stats import mannwhitneyu
from matplotlib.lines import Line2D

# ============================================================
# 1. 参数
# ============================================================
output_dir = "./GSE306542_clinical_analysis"
os.makedirs(output_dir, exist_ok=True)

response_col = "early_response_9weeks"
font_dir = str(FONT_DIR)

for f in os.listdir(font_dir):
    if f.lower().endswith((".ttf", ".otf")):
        fm.fontManager.addfont(os.path.join(font_dir, f))

mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "font.size": 15,
    "font.weight": "bold",
    "axes.labelsize": 17,
    "axes.labelweight": "bold",
    "xtick.labelsize": 15,
    "ytick.labelsize": 14,
    "axes.linewidth": 1.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.transparent": True
})

# ============================================================
# 2. 数据准备
# ============================================================
df = prediction_clinical_df.copy()

cols = ["Actual_Age", "Predicted_Age", "Age_Diff", response_col]
plot_df = df[cols].copy()

for col in ["Actual_Age", "Predicted_Age", "Age_Diff"]:
    plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")

plot_df = plot_df.dropna(subset=cols).copy()
plot_df["Response"] = plot_df[response_col].astype(str)

order = ["No", "Yes"]
if set(plot_df["Response"]) != set(order):
    raise ValueError(f"Unexpected response groups: {plot_df['Response'].unique()}")

color_map = {
    "No": "#668DAA",
    "Yes": "#C98275"
}

label_map = {
    "No": "Non-responder",
    "Yes": "Responder"
}

# ============================================================
# 3. 统计检验
# ============================================================
actual_no = plot_df.loc[plot_df["Response"] == "No", "Actual_Age"].to_numpy()
actual_yes = plot_df.loc[plot_df["Response"] == "Yes", "Actual_Age"].to_numpy()

diff_no = plot_df.loc[plot_df["Response"] == "No", "Age_Diff"].to_numpy()
diff_yes = plot_df.loc[plot_df["Response"] == "Yes", "Age_Diff"].to_numpy()

# 实际年龄组间比较
u_age, p_age = mannwhitneyu(
    actual_no, actual_yes,
    alternative="two-sided",
    method="exact"
)

# Age_Diff组间比较（这里仅打印，图中不标注）
u_diff, p_diff = mannwhitneyu(
    diff_no, diff_yes,
    alternative="two-sided",
    method="exact"
)

print("===== Actual age =====")
print(f"Non-responder: n={len(actual_no)}, median={np.median(actual_no):.2f}")
print(f"Responder: n={len(actual_yes)}, median={np.median(actual_yes):.2f}")
print(f"Mann–Whitney U={u_age:.1f}, P={p_age:.4f}")

print("\n===== Age_Diff =====")
print(f"Non-responder: median={np.median(diff_no):.2f}")
print(f"Responder: median={np.median(diff_yes):.2f}")
print(f"Mann–Whitney U={u_diff:.1f}, P={p_diff:.4f}")

# ============================================================
# 4. 绘制配对图
# ============================================================
fig, ax = plt.subplots(figsize=(5.0, 5.1))

rng = np.random.default_rng(42)
x_pos = [0, 1]

# 让不同响应组略微水平错开
group_offset = {"No": -0.12, "Yes": 0.12}

for _, row in plot_df.iterrows():
    group = row["Response"]
    color = color_map[group]

    offset = group_offset[group] + rng.uniform(-0.035, 0.035)

    x1 = x_pos[0] + offset
    x2 = x_pos[1] + offset

    # 同一个患者的配对连线
    ax.plot(
        [x1, x2],
        [row["Actual_Age"], row["Predicted_Age"]],
        color=color,
        linewidth=1.6,
        alpha=0.72,
        zorder=1
    )

    # 实际年龄
    ax.scatter(
        x1, row["Actual_Age"],
        s=85, color=color,
        edgecolor="#333333",
        linewidth=0.8,
        zorder=3
    )

    # 预测年龄
    ax.scatter(
        x2, row["Predicted_Age"],
        s=85, color=color,
        edgecolor="#333333",
        linewidth=0.8,
        zorder=3
    )

# ============================================================
# 5. 中位数横线
# ============================================================
median_actual = plot_df["Actual_Age"].median()
median_pred = plot_df["Predicted_Age"].median()

ax.hlines(
    median_actual, -0.23, 0.23,
    color="#333333", linewidth=2.5, zorder=4
)
ax.hlines(
    median_pred, 0.77, 1.23,
    color="#333333", linewidth=2.5, zorder=4
)

# ============================================================
# 6. 左侧实际年龄组间 P 值
# ============================================================
ymax_actual = plot_df["Actual_Age"].max()
ymin_all = min(plot_df["Actual_Age"].min(), plot_df["Predicted_Age"].min())
ymax_all = max(plot_df["Actual_Age"].max(), plot_df["Predicted_Age"].max())

yrange = ymax_all - ymin_all
bracket_y = ymax_actual + yrange * 0.075
bracket_h = yrange * 0.025

# P值括号，只放在 Actual age 上方
ax.plot(
    [-0.17, -0.17, 0.17, 0.17],
    [bracket_y, bracket_y + bracket_h,
     bracket_y + bracket_h, bracket_y],
    color="#333333",
    linewidth=1.5,
    clip_on=False
)

ax.text(
    0, bracket_y + bracket_h + yrange * 0.025,
    f"P = {p_age:.3f}",
    ha="center", va="bottom",
    fontsize=15,
    fontweight="bold"
)

# ============================================================
# 7. 坐标轴与图例
# ============================================================
ax.set_xticks(x_pos)
ax.set_xticklabels(
    ["Actual age", "Predicted age"],
    fontsize=15,
    fontweight="bold"
)

ax.set_ylabel("Age (years)", fontsize=17, fontweight="bold", labelpad=8)

# 图例放在图内左上角，减少外部留白
handles = [
    Line2D([0], [0], marker="o", linestyle="-",
           color=color_map[g], markerfacecolor=color_map[g],
           markeredgecolor="#333333", linewidth=1.6,
           markersize=8, label=f"{label_map[g]} (n={sum(plot_df['Response'] == g)})")
    for g in order
]

ax.legend(
    handles=handles,
    loc="upper left",
    bbox_to_anchor=(0.01, 0.99),
    frameon=False,
    fontsize=11,
    prop={"family": "Arial", "weight": "bold", "size": 11},
    handlelength=1.7
)

ax.set_xlim(-0.43, 1.43)
ax.set_ylim(
    ymin_all - yrange * 0.08,
    max(ymax_all, bracket_y + bracket_h) + yrange * 0.10
)

# ============================================================
# 8. 样式
# ============================================================
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(1.5)
ax.spines["bottom"].set_linewidth(1.5)

ax.tick_params(
    axis="both", which="major",
    direction="out", width=1.4, length=5, pad=5
)

for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontweight("bold")

ax.grid(False)
fig.subplots_adjust(left=0.20, right=0.97, bottom=0.13, top=0.97)

# ============================================================
# 9. 保存
# ============================================================
plt.savefig(
    f"{output_dir}/Actual_vs_Predicted_Age_byResponse.pdf",
    bbox_inches="tight", pad_inches=0.04
)
plt.savefig(
    f"{output_dir}/Actual_vs_Predicted_Age_byResponse.png",
    dpi=600, bbox_inches="tight", pad_inches=0.04
)

plt.show()

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
from scipy.stats import mannwhitneyu

# ============================================================
# 1. 参数
# ============================================================
output_dir = "./GSE306542_clinical_analysis"
os.makedirs(output_dir, exist_ok=True)

response_col = "early_response_9weeks"
age_col = "Age_Diff"

# Arial 字体
font_dir = str(FONT_DIR)
for f in os.listdir(font_dir):
    if f.lower().endswith((".ttf", ".otf")):
        fm.fontManager.addfont(os.path.join(font_dir, f))

mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "font.size": 15,
    "font.weight": "bold",
    "axes.labelsize": 17,
    "axes.labelweight": "bold",
    "axes.titlesize": 16,
    "axes.titleweight": "bold",
    "xtick.labelsize": 15,
    "ytick.labelsize": 14,
    "axes.linewidth": 1.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.transparent": True
})

# ============================================================
# 2. 整理数据
# ============================================================
df = prediction_clinical_df.copy()
df[age_col] = pd.to_numeric(df[age_col], errors="coerce")
df = df.dropna(subset=[response_col, age_col]).copy()
df["Response"] = df[response_col].astype(str)

groups = sorted(df["Response"].unique())
if len(groups) != 2:
    raise ValueError(f"Expected 2 response groups, found: {groups}")

# 如果已经确认两个组的编码，可以在这里指定顺序
order = groups

data = [df.loc[df["Response"] == g, age_col].to_numpy() for g in order]
counts = [len(x) for x in data]
medians = [np.median(x) for x in data]

print("\n===== Group summary =====")
for g, values in zip(order, data):
    print(f"{g}: n={len(values)}, median={np.median(values):.2f}, mean={np.mean(values):.2f}")

# Mann–Whitney U 检验（双侧）
p_value = np.nan
if min(counts) >= 2:
    _, p_value = mannwhitneyu(data[0], data[1], alternative="two-sided", method="auto")

print(f"\nMann–Whitney P = {p_value:.4g}")
print(f"Median difference ({order[1]} - {order[0]}): {medians[1] - medians[0]:.2f} years")

# ============================================================
# 3. 绘图
# ============================================================
fig, ax = plt.subplots(figsize=(4.5, 4.6))

colors = ["#668DAA", "#C98275"]
positions = [0, 1]

# 箱线图
bp = ax.boxplot(
    data,
    positions=positions,
    widths=0.40,
    patch_artist=True,
    showfliers=False,
    showmeans=False,
    boxprops=dict(linewidth=1.5, edgecolor="#333333"),
    medianprops=dict(color="#242424", linewidth=2.4),
    whiskerprops=dict(color="#444444", linewidth=1.5),
    capprops=dict(color="#444444", linewidth=1.5)
)

for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.35)

# 每个患者的散点
rng = np.random.default_rng(42)

for i, values in enumerate(data):
    jitter = rng.uniform(-0.12, 0.12, len(values))
    ax.scatter(
        np.full(len(values), positions[i]) + jitter,
        values,
        s=75,
        color=colors[i],
        edgecolor="#333333",
        linewidth=0.8,
        alpha=0.95,
        zorder=4
    )

# Age_Diff = 0 参考线
ax.axhline(
    0, color="#888888",
    linestyle="--", linewidth=1.2,
    zorder=0
)

# ============================================================
# 4. 中位数标签
# ============================================================
for i, median in enumerate(medians):
    ax.annotate(
        f"{median:+.1f}",
        xy=(positions[i] + 0.25, median),
        xytext=(5, 0),
        textcoords="offset points",
        fontsize=13,
        fontweight="bold",
        color=colors[i],
        va="center",
        ha="left"
    )

# ============================================================
# 5. 坐标轴与统计结果
# ============================================================
ax.set_xticks(positions)
ax.set_xticklabels(
    [f"{g}\n(n={n})" for g, n in zip(order, counts)],
    fontweight="bold"
)

ax.set_ylabel("Predicted age − Actual age (years)", fontweight="bold", labelpad=8)
ax.set_xlabel("")

# 统计结果放在标题位置，避免占用绘图区
p_text = f"P = {p_value:.3g}" if np.isfinite(p_value) else "P = NA"
ax.set_title(p_text, fontsize=16, fontweight="bold", pad=9)

# 适当压缩X轴留白
ax.set_xlim(-0.55, 1.65)

# Y轴覆盖全部数据，保留少量上下边距
ymin = min(np.min(x) for x in data)
ymax = max(np.max(x) for x in data)
yrange = max(ymax - ymin, 1)

ax.set_ylim(
    min(ymin, 0) - yrange * 0.08,
    max(ymax, 0) + yrange * 0.08
)

# 坐标轴样式
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_linewidth(1.5)
ax.spines["bottom"].set_linewidth(1.5)

ax.tick_params(
    axis="both",
    which="major",
    direction="out",
    width=1.4,
    length=5,
    pad=5
)

for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontweight("bold")

ax.grid(False)

# 尽量减少外边距
fig.subplots_adjust(left=0.24, right=0.97, bottom=0.17, top=0.90)

# ============================================================
# 6. 保存
# ============================================================
plt.savefig(
    f"{output_dir}/AgeDiff_by_response.pdf",
    bbox_inches="tight",
    pad_inches=0.04
)

plt.savefig(
    f"{output_dir}/AgeDiff_by_response.png",
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.04
)

plt.show()